# S1 · Visualizacion con Python sobre un mismo dataset

Este notebook esta pensado para clase.

La logica es:
- cargamos un solo dataset
- planteamos una sola historia
- resolvemos varias preguntas visuales
- comparamos como se ve en distintas librerias
- despues conectamos eso con Streamlit


## 1. Cargar librerias y dataset

### Que hace este bloque

- `pandas` sirve para trabajar tablas
- `matplotlib` sirve para la base del grafico
- `seaborn` mejora estilo y facilidad
- `plotly.express` da interactividad

Aqui tambien configuramos un estilo base con `seaborn` y un tamano inicial para las figuras.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

df = px.data.gapminder().copy()
df.head()

### Que significa `df`

`df` es la tabla principal de toda la clase.

Estas lineas ayudan a revisar:
- cuantas filas hay
- cuantas columnas hay
- como se llaman las variables


In [ ]:
df.shape, df.columns.tolist()

## 2. Contexto del caso

### Pregunta general

> Como ha cambiado el desarrollo humano entre 1952 y 2007, que regiones avanzan mas lento y donde deberiamos mirar primero?

Para varias visualizaciones vamos a trabajar con una fotografia del ano 2007.

### Que hace este bloque

- `year_selected` guarda el ano elegido
- `df_2007` crea una nueva tabla solo con ese ano

`df` = todos los anos

`df_2007` = solo el ano 2007


In [ ]:
year_selected = 2007
df_2007 = df[df['year'] == year_selected].copy()
df_2007.head()

## 3. Correlation

### Pregunta visual

> Hay relacion entre PIB per capita y esperanza de vida?

Cuando queremos comparar dos variables numericas, el grafico mas natural es un `scatter plot`.


### Matplotlib

#### Que hace este codigo

- `ax.scatter(...)` dibuja puntos
- `ax.set_xscale('log')` pone el eje X en escala logaritmica
- `set_title`, `set_xlabel`, `set_ylabel` cambian titulos y etiquetas


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(df_2007['gdpPercap'], df_2007['lifeExp'], alpha=0.7, color='steelblue')
ax.set_xscale('log')
ax.set_title('Correlation · Matplotlib')
ax.set_xlabel('PIB per capita (log)')
ax.set_ylabel('Esperanza de vida')
plt.show()

### Seaborn

#### Que hace este codigo

- `data=df_2007` indica de donde salen los datos
- `x` y `y` definen los ejes
- `hue='continent'` cambia el color por continente
- `size='pop'` cambia el tamano por poblacion


In [ ]:
plt.figure(figsize=(10, 6))
sns.scatterplot(
    data=df_2007,
    x='gdpPercap',
    y='lifeExp',
    hue='continent',
    size='pop',
    sizes=(20, 300),
    alpha=0.7
)
plt.xscale('log')
plt.title('Correlation · Seaborn')
plt.xlabel('PIB per capita (log)')
plt.ylabel('Esperanza de vida')
plt.show()

### Plotly

#### Que hace este codigo

- `color='continent'` pinta por categoria
- `size='pop'` cambia tamano
- `hover_name='country'` muestra el pais al pasar el mouse
- `log_x=True` deja el eje X en escala logaritmica


In [ ]:
fig = px.scatter(
    df_2007,
    x='gdpPercap',
    y='lifeExp',
    color='continent',
    size='pop',
    hover_name='country',
    log_x=True,
    title='Correlation · Plotly'
)
fig.show()

## 4. Ranking

### Pregunta visual

> Que paises estan arriba y abajo en esperanza de vida?

Antes de graficar, hay que ordenar.


In [ ]:
ranking = df_2007[['country', 'lifeExp']].sort_values('lifeExp', ascending=False).head(15)
ranking.tail()

### Matplotlib

#### Que hace este codigo

- `ranking_sorted` reordena para que la barra horizontal se lea mejor
- `barh` crea barras horizontales
- `color` cambia el color


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
ranking_sorted = ranking.sort_values('lifeExp')
ax.barh(ranking_sorted['country'], ranking_sorted['lifeExp'], color='#2F8373')
ax.set_title('Ranking · Matplotlib')
ax.set_xlabel('Esperanza de vida')
ax.set_ylabel('Pais')
plt.show()

### Seaborn

Aqui `palette='viridis'` cambia la paleta de color.


In [ ]:
plt.figure(figsize=(10, 7))
sns.barplot(data=ranking_sorted, x='lifeExp', y='country', palette='viridis')
plt.title('Ranking · Seaborn')
plt.xlabel('Esperanza de vida')
plt.ylabel('Pais')
plt.show()

### Plotly

En Plotly el ranking queda interactivo.


In [ ]:
fig = px.bar(
    ranking_sorted,
    x='lifeExp',
    y='country',
    orientation='h',
    title='Ranking · Plotly'
)
fig.show()

## 5. Distribution

### Pregunta visual

> Como se distribuye la esperanza de vida? Hay diferencias por continente?


### Matplotlib

Usamos un histograma para ver forma general, frecuencia y dispersion.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(df_2007['lifeExp'], bins=20, color='tomato', edgecolor='white')
ax.set_title('Distribution · Matplotlib')
ax.set_xlabel('Esperanza de vida')
ax.set_ylabel('Frecuencia')
plt.show()

### Seaborn

Aqui usamos `boxplot` para comparar grupos.


In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_2007, x='continent', y='lifeExp')
plt.title('Distribution · Seaborn')
plt.xlabel('Continente')
plt.ylabel('Esperanza de vida')
plt.show()

### Plotly

Este histograma deja explorar con hover y color por continente.


In [ ]:
fig = px.histogram(
    df_2007,
    x='lifeExp',
    color='continent',
    nbins=20,
    title='Distribution · Plotly'
)
fig.show()

## 6. Change over Time

### Pregunta visual

> Como ha cambiado la esperanza de vida promedio por continente entre 1952 y 2007?

Antes de dibujar la linea, necesitamos resumir datos.


In [ ]:
serie_tiempo = (
    df.groupby(['year', 'continent'], as_index=False)['lifeExp']
      .mean()
)
serie_tiempo.head()

### Matplotlib

Hacemos una linea por continente.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for continente, grupo in serie_tiempo.groupby('continent'):
    ax.plot(grupo['year'], grupo['lifeExp'], marker='o', label=continente)
ax.set_title('Change over Time · Matplotlib')
ax.set_xlabel('Ano')
ax.set_ylabel('Esperanza de vida promedio')
ax.legend()
plt.show()

### Seaborn

Seaborn simplifica bastante el grafico temporal.


In [ ]:
plt.figure(figsize=(10, 6))
sns.lineplot(data=serie_tiempo, x='year', y='lifeExp', hue='continent', marker='o')
plt.title('Change over Time · Seaborn')
plt.xlabel('Ano')
plt.ylabel('Esperanza de vida promedio')
plt.show()

### Plotly

Plotly mantiene la misma logica, pero con interactividad.


In [ ]:
fig = px.line(
    serie_tiempo,
    x='year',
    y='lifeExp',
    color='continent',
    markers=True,
    title='Change over Time · Plotly'
)
fig.show()

## 7. Que pueden cambiar sin miedo

Cambios seguros:
- el color
- el titulo
- la variable del eje X
- la variable del eje Y
- el ano seleccionado
- el top 15 por top 10

La idea no es escribir un proyecto nuevo.
La idea es tocar algo pequeno y ver el efecto.


## 8. Puente hacia Streamlit

Hasta aqui vimos graficos sueltos.

El siguiente paso natural es:
- poner filtros
- recalcular automaticamente
- dejar que otra persona explore la informacion

Eso es exactamente lo que hace Streamlit.

Frase docente sugerida:

`Con el notebook pensamos y comparamos.`

`Con Streamlit envolvemos ese analisis en una app.`
